# PCU-KILL-001 — Granite Engineering E0 / Context Oracle v2

Engineering-only Kaggle runner for `ibm-granite/granite-3.1-1b-a400m-base`.

This notebook preserves the original published `26090501` testbed-kill result and writes the repaired run to `26090501-oracle-v2`. Formal seeds are never executed here.

Requirements: Kaggle Internet ON, GPU enabled, Secrets `HF_TOKEN` and `GITHUB_TOKEN`.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

BRANCH = "codex/pcu-composability-kill-001"
REPO = Path("/kaggle/working/mini-cells")
ENGINEERING_SEED = 26090501
RUN_ID = "26090501-oracle-v2"
OUT = REPO / "artifacts/research/pcu-kill-001/engineering" / RUN_ID
MODEL_ID = "ibm-granite/granite-3.1-1b-a400m-base"
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = "5.16.1"

os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    p = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return p.stdout.strip() if capture else ""

if REPO.exists():
    shutil.rmtree(REPO)
run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])
os.chdir(REPO)
run(["git", "pull", "--ff-only", "origin", BRANCH])

assert sys.version_info >= (3, 11)
run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])
run([sys.executable, "-m", "pip", "install",
     f"transformers=={REQUIRED_TRANSFORMERS}",
     "huggingface_hub>=0.36,<2.0", "safetensors>=0.4", "accelerate>=1.0"])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."

SOURCE_COMMIT = run(["git", "rev-parse", "HEAD"], capture=True)
SOURCE_TREE = run(["git", "rev-parse", "HEAD^{tree}"], capture=True)
print(json.dumps({
    "branch": run(["git", "branch", "--show-current"], capture=True),
    "commit": SOURCE_COMMIT,
    "tree": SOURCE_TREE,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "run_id": RUN_ID,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
github_token = secrets.get_secret("GITHUB_TOKEN")
assert hf_token, "Missing Kaggle Secret: HF_TOKEN"
assert github_token, "Missing Kaggle Secret: GITHUB_TOKEN"

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["GITHUB_TOKEN"] = github_token
login(token=hf_token, add_to_git_credential=False)
print("Kaggle Secrets loaded; token values were not printed.")


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoConfig

info = HfApi(token=os.environ["HF_TOKEN"]).model_info(MODEL_ID)
HF_REVISION = info.sha
cfg = AutoConfig.from_pretrained(MODEL_ID, revision=HF_REVISION, token=os.environ["HF_TOKEN"])

assert cfg.model_type == "granitemoe"
assert int(cfg.hidden_size) == 1024
assert int(cfg.intermediate_size) == 512
assert int(cfg.num_hidden_layers) == 24
assert int(cfg.num_local_experts) == 32
assert int(cfg.num_experts_per_tok) == 8

print(json.dumps({
    "model": MODEL_ID,
    "hf_revision": HF_REVISION,
    "hidden_size": cfg.hidden_size,
    "intermediate_size": cfg.intermediate_size,
    "layers": cfg.num_hidden_layers,
    "experts": cfg.num_local_experts,
    "top_k": cfg.num_experts_per_tok,
    "logits_scaling": getattr(cfg, "logits_scaling", None),
}, indent=2))


In [ ]:
SEED_REGISTRY = REPO / "research/formal_seed_registry.json"

def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(x["seed"]): x["state"] for x in payload["seeds"]}

expected = {seed: "RESERVED_UNTOUCHED" for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(["git", "diff", "--", str(SEED_REGISTRY.relative_to(REPO))], capture=True) == ""

old = REPO / "artifacts/research/pcu-kill-001/engineering/26090501/ENGINEERING_DECISION.json"
assert old.is_file(), "Expected the published v1 testbed-kill evidence to remain in history."
probe = subprocess.run(
    ["git", "ls-files", "--error-unmatch", str((OUT / "ENGINEERING_DECISION.json").relative_to(REPO))],
    cwd=REPO, text=True, capture_output=True,
)
assert probe.returncode != 0, f"{RUN_ID} is already tracked; refusing to overwrite it."
assert not (REPO / "artifacts/research/pcu-kill-001/formal").exists()

print(json.dumps({
    "engineering_seed": ENGINEERING_SEED,
    "run_id": RUN_ID,
    "formal_seed_states": formal_states(),
    "previous_e0_preserved": True,
}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(REPO / "src")
test_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

run([sys.executable, "-m", "pytest", "-q", "tests/research/05-pcu-kill-001"], env=test_env)
run([sys.executable, "-m", "compileall", "-q", "src/minicells/pcu_kill_001", "scripts/research"])
print("PCU-KILL-001 test/compile gate: PASS")


In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)

run([
    sys.executable, "scripts/research/run_pcu_kill_001.py",
    "--phase", "engineering",
    "--backend", "granite",
    "--seed", str(ENGINEERING_SEED),
    "--device", "cuda",
    "--out", OUT,
])


In [ ]:
required = [
    "RUN_IDENTITY.json", "MODEL_MANIFEST.json", "DATASET_MANIFEST.json",
    "DATASET_AUDIT.json", "EQUIVALENCE.json", "CACHE_EQUIVALENCE.json",
    "CONTEXT_ORACLE.json", "ENGINEERING_DECISION.json",
]
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing

decision = json.loads((OUT / "ENGINEERING_DECISION.json").read_text())
oracle = json.loads((OUT / "CONTEXT_ORACLE.json").read_text())
equiv = json.loads((OUT / "EQUIVALENCE.json").read_text())
identity = json.loads((OUT / "RUN_IDENTITY.json").read_text())

assert identity["run_id"] == RUN_ID
assert identity["seed"] == ENGINEERING_SEED
assert identity["positive_control_version"] == "pcu-kill-001-context-oracle-v2"
assert identity["source"]["source_dirty"] is False
assert decision["phase"] == "engineering"
assert decision["scientific_evidence"] is False
assert decision["formal_execution_not_started"] is True
assert equiv["g0_exact_embedding"] is True
assert equiv["cache"]["passed"] is True

print(json.dumps({
    "status": decision.get("status"),
    "formal_ready": decision.get("formal_ready"),
    "g0": equiv["g0_exact_embedding"],
    "cache": equiv["cache"]["passed"],
    "oracle": {
        "passed": oracle.get("passed"),
        "retrieval_a_accuracy": oracle.get("retrieval_a_accuracy"),
        "retrieval_b_accuracy": oracle.get("retrieval_b_accuracy"),
        "composition_accuracy": oracle.get("composition_accuracy"),
        "free_generation_diagnostic_accuracy": (oracle.get("free_generation_diagnostic") or {}).get("accuracy"),
    },
    "source_dirty": identity["source"]["source_dirty"],
}, indent=2))


In [ ]:
assert formal_states() == expected
assert run(["git", "diff", "--", str(SEED_REGISTRY.relative_to(REPO))], capture=True) == ""

run([
    sys.executable,
    "scripts/research/publish_pcu_kill_001_engineering.py",
    "--branch", BRANCH,
    "--run-id", RUN_ID,
])

assert formal_states() == expected
print(json.dumps({
    "published_run_id": RUN_ID,
    "formal_seed_states": formal_states(),
    "formal_execution_not_started": True,
}, indent=2))
